In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
file_path = '/content/drive/MyDrive/CNLP_LAB/exp-4/4.1_4.2_input.txt'

with open(file_path, 'r', encoding='utf-8') as f:
    text_content = f.read()

print(f"Content fetched directly from '{file_path}'.\n")
print(text_content[:500])

Content fetched directly from '/content/drive/MyDrive/CNLP_LAB/exp-4/4.1_4.2_input.txt'.

Natural Language Processing and Artificial Intelligence

Natural Language Processing, commonly known as NLP, is an important area of artificial intelligence that focuses on enabling computers to understand, process, analyze, and generate human language. Human beings communicate through languages such as English, Hindi, Assamese, Bengali, Tamil, Telugu, Marathi, and many other languages. Computers, however, process information in numerical and symbolic forms. NLP provides methods that help bridge


## 4.1. Term-Frequency Analysis and Named Entity Recognition (with toolkit)

First, we'll install `nltk` and `spaCy` and download their necessary data models. Then, we'll perform tokenization, calculate word frequencies, and identify named entities.

In [5]:
import nltk
import spacy
from collections import Counter
import pandas as pd
import os

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

try:
    nlp = spacy.load('en_core_web_sm')
except OSError:
    print('Downloading spaCy model (en_core_web_sm). This may take a moment...')
    os.system('python -m spacy download en_core_web_sm')
    nlp = spacy.load('en_core_web_sm')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [6]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import re

# Preprocessing for TF analysis
cleaned_text = re.sub(r'[^a-zA-Z\s]', '', text_content.lower())
words = word_tokenize(cleaned_text)
stop_words = set(stopwords.words('english'))
filtered_words = [word for word in words if word.isalpha() and word not in stop_words]

# Calculate frequency
word_counts = Counter(filtered_words)

# Save to CSV
tf_df_toolkit = pd.DataFrame(word_counts.items(), columns=['Term', 'Frequency'])
tf_df_toolkit.to_csv('term_frequency_toolkit.csv', index=False)
print("Term frequencies saved to 'term_frequency_toolkit.csv'")

# Identify and display top 10 most frequent terms
top_10_terms_toolkit = tf_df_toolkit.sort_values(by='Frequency', ascending=False).head(10)
print("\nTop 10 most frequent terms (with toolkit):\n")
display(top_10_terms_toolkit)

Term frequencies saved to 'term_frequency_toolkit.csv'

Top 10 most frequent terms (with toolkit):



,Term,Frequency
86,may,30
127,word,30
30,information,28
1,language,25
81,words,25
94,text,24
99,document,23
41,systems,22
7,nlp,21
139,analysis,18


In [7]:
# Perform Named Entity Recognition (NER) with spaCy
doc = nlp(text_content)

print("\nNamed Entities:\n")
for ent in doc.ents:
    print(f"Text: {ent.text}, Label: {ent.label_}, Explanation: {spacy.explain(ent.label_)}")


Named Entities:

Text: Natural Language Processing and Artificial Intelligence

Natural Language Processing, Label: ORG, Explanation: Companies, agencies, institutions, etc.
Text: NLP, Label: ORG, Explanation: Companies, agencies, institutions, etc.
Text: English, Label: LANGUAGE, Explanation: Any named language
Text: Hindi, Label: GPE, Explanation: Countries, cities, states
Text: Assamese, Label: NORP, Explanation: Nationalities or religious or political groups
Text: Bengali, Label: NORP, Explanation: Nationalities or religious or political groups
Text: Tamil, Label: GPE, Explanation: Countries, cities, states
Text: NLP, Label: ORG, Explanation: Companies, agencies, institutions, etc.
Text: NLP, Label: ORG, Explanation: Companies, agencies, institutions, etc.
Text: NLP, Label: ORG, Explanation: Companies, agencies, institutions, etc.
Text: twenty, Label: CARDINAL, Explanation: Numerals that do not fall under another type
Text: twenty, Label: DATE, Explanation: Absolute or relative da

## 4.2. Term-Frequency Analysis (without toolkit)

Now, we'll manually implement tokenization, clean the text by converting to lowercase and removing punctuation, and then calculate word frequencies.

In [8]:
import string

# Manual Preprocessing (lowercase, remove punctuation, tokenize)
def preprocess_manual(text):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    words = text.split()
    return [word for word in words if word.isalpha()]

# Calculate frequency manually
def calculate_term_frequency_manual(words):
    freq = {}
    for word in words:
        freq[word] = freq.get(word, 0) + 1
    return freq

processed_words_manual = preprocess_manual(text_content)
word_counts_manual = calculate_term_frequency_manual(processed_words_manual)

# Save to CSV
tf_df_manual = pd.DataFrame(word_counts_manual.items(), columns=['Term', 'Frequency'])
tf_df_manual.to_csv('term_frequency_manual.csv', index=False)
print("Term frequencies (manual) saved to 'term_frequency_manual.csv'")

# Identify and display top 10 most frequent terms
top_10_terms_manual = tf_df_manual.sort_values(by='Frequency', ascending=False).head(10)
print("\nTop 10 most frequent terms (without toolkit):\n")
display(top_10_terms_manual)

Term frequencies (manual) saved to 'term_frequency_manual.csv'

Top 10 most frequent terms (without toolkit):



,Term,Frequency
97,a,90
3,and,86
50,the,75
20,to,42
42,in,42
100,can,41
14,of,40
10,is,37
158,word,30
112,may,30


## 4.3. TF-IDF Calculation (without NLTK, spaCy, or any other NLP toolkit)

For this section, we will define three sample documents as specified and implement all the necessary functions from scratch to calculate TF, DF, IDF, and TF-IDF.

In [9]:
import math

document1 = "Natural language processing is a field of artificial intelligence."
document2 = "Natural language processing helps computers understand human language."
document3 = "Machine learning is an important part of artificial intelligence."

documents = {
    'Document 1': document1,
    'Document 2': document2,
    'Document 3': document3
}

def to_lowercase(text):
    return text.lower()

In [10]:
# Function to remove punctuation
def remove_punctuation(text):
    return text.translate(str.maketrans('', '', string.punctuation))

processed_documents_no_punct = {}
for doc_name, doc_text in documents.items():
    processed_text = remove_punctuation(to_lowercase(doc_text))
    processed_documents_no_punct[doc_name] = processed_text

In [11]:
# Function to tokenize the documents
def tokenize_document(text):
    return [word for word in text.split() if word.strip()]

tokenized_documents = {}
for doc_name, doc_text in processed_documents_no_punct.items():
    tokens = tokenize_document(doc_text)
    tokenized_documents[doc_name] = tokens
print("\n--- Tokenized Documents (Example for Document 1) ---")
print(f"Document 1: {tokenized_documents['Document 1']}")


--- Tokenized Documents (Example for Document 1) ---
Document 1: ['natural', 'language', 'processing', 'is', 'a', 'field', 'of', 'artificial', 'intelligence']


In [12]:
# Function to calculate TF for every term
def calculate_tf(tokens):
    tf_dict = {}
    total_terms = len(tokens)
    if total_terms == 0:
        return tf_dict
    for term in tokens:
        tf_dict[term] = tf_dict.get(term, 0) + 1
    for term in tf_dict:
        tf_dict[term] = tf_dict[term] / total_terms
    return tf_dict

tf_scores = {}
for doc_name, tokens in tokenized_documents.items():
    tf_scores[doc_name] = calculate_tf(tokens)
print("\n--- TF Scores (Example for Document 1) ---")
for term, tf in tf_scores['Document 1'].items():
    print(f"  {term}: {tf:.4f}")


--- TF Scores (Example for Document 1) ---
  natural: 0.1111
  language: 0.1111
  processing: 0.1111
  is: 0.1111
  a: 0.1111
  field: 0.1111
  of: 0.1111
  artificial: 0.1111
  intelligence: 0.1111


In [13]:
# Function to calculate DF for every term
def calculate_df(all_tokenized_documents):
    df_dict = {}
    for tokens in all_tokenized_documents.values():
        unique_terms_in_doc = set(tokens)
        for term in unique_terms_in_doc:
            df_dict[term] = df_dict.get(term, 0) + 1
    return df_dict

df_scores = calculate_df(tokenized_documents)
print("\n--- DF Scores ---")
for term, df in df_scores.items():
    print(f"  {term}: {df}")


--- DF Scores ---
  natural: 2
  intelligence: 2
  language: 2
  is: 2
  of: 2
  artificial: 2
  field: 1
  a: 1
  processing: 2
  computers: 1
  human: 1
  understand: 1
  helps: 1
  an: 1
  important: 1
  part: 1
  learning: 1
  machine: 1


In [14]:
# Function to calculate IDF using the logarithmic formula
def calculate_idf(df_scores, total_documents):
    idf_dict = {}
    for term, df in df_scores.items():
        idf_dict[term] = math.log10(total_documents / df)
    return idf_dict

total_documents = len(documents)
idf_scores = calculate_idf(df_scores, total_documents)
print("\n--- IDF Scores ---")
for term, idf in idf_scores.items():
    print(f"  {term}: {idf:.4f}")


--- IDF Scores ---
  natural: 0.1761
  intelligence: 0.1761
  language: 0.1761
  is: 0.1761
  of: 0.1761
  artificial: 0.1761
  field: 0.4771
  a: 0.4771
  processing: 0.1761
  computers: 0.4771
  human: 0.4771
  understand: 0.4771
  helps: 0.4771
  an: 0.4771
  important: 0.4771
  part: 0.4771
  learning: 0.4771
  machine: 0.4771


In [15]:
# Function to calculate TF-IDF for every term in every document
def calculate_tfidf(tf_scores, idf_scores):
    tfidf_scores = {}
    for doc_name, tf_dict in tf_scores.items():
        tfidf_doc = {}
        for term, tf in tf_dict.items():
            tfidf_doc[term] = tf * idf_scores.get(term, 0) # Use .get for terms not in idf_scores
        tfidf_scores[doc_name] = tfidf_doc
    return tfidf_scores

tfidf_scores = calculate_tfidf(tf_scores, idf_scores)
print("\n--- TF-IDF Scores (Example for Document 1) ---")
for term, tfidf in tfidf_scores['Document 1'].items():
    print(f"  {term}: {tfidf:.4f}")


--- TF-IDF Scores (Example for Document 1) ---
  natural: 0.0196
  language: 0.0196
  processing: 0.0196
  is: 0.0196
  a: 0.0530
  field: 0.0530
  of: 0.0196
  artificial: 0.0196
  intelligence: 0.0196


In [16]:
# Identify the top 10 terms with the highest TF-IDF score for each document
print("\n--- Top 10 terms with highest TF-IDF score for each document ---")
for doc_name, tfidf_doc in tfidf_scores.items():
    # Sort terms by TF-IDF score in descending order
    sorted_terms = sorted(tfidf_doc.items(), key=lambda item: item[1], reverse=True)
    print(f"\n{doc_name}:")
    for term, score in sorted_terms[:10]:
        print(f"  {term}: {score:.4f}")


--- Top 10 terms with highest TF-IDF score for each document ---

Document 1:
  a: 0.0530
  field: 0.0530
  natural: 0.0196
  language: 0.0196
  processing: 0.0196
  is: 0.0196
  of: 0.0196
  artificial: 0.0196
  intelligence: 0.0196

Document 2:
  helps: 0.0596
  computers: 0.0596
  understand: 0.0596
  human: 0.0596
  language: 0.0440
  natural: 0.0220
  processing: 0.0220

Document 3:
  machine: 0.0530
  learning: 0.0530
  an: 0.0530
  important: 0.0530
  part: 0.0530
  is: 0.0196
  of: 0.0196
  artificial: 0.0196
  intelligence: 0.0196
